In [1]:
import json

with open("knowledge_base.json", "r") as f:
    docs = json.load(f)

print(docs[0])

{'id': 'kb-01', 'source': 'handbook.md', 'text': 'Employees may park in lot B after 6pm on weekdays. Lot A is reserved for visitors during business hours and requires a pass from reception.'}


In [3]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="company_kb"
)

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)



/opt/anaconda3/envs/ironhack/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9695.75it/s]


In [4]:
for doc in docs:
    embedding = embedding_model.encode(doc["text"]).tolist()
    collection.add(
        ids=[doc["id"]],
        documents=[doc["text"]],
        embeddings=[embedding],
        metadatas=[
            {"source": doc["source"]}
        ]
    )

# No chunking is needed because each entry in the
# knowledge base is already a small self-contained passage.
# For larger documents we would split them into chunks
# before embedding to improve retrieval quality.

def retrieve(question):
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    return results

In [10]:
def build_prompt(question, results):

    passages = []

    docs = results["documents"][0]
    metadata = results["metadatas"][0]

    for doc, meta in zip(docs, metadata):

        passages.append(
            f"Source: {meta['source']}\n"
            f"Text: {doc}"
        )

    context = "\n\n".join(passages)

    prompt = f"""You are a precise corporate assistant. Answer the question using ONLY the context provided below.
    
CRITICAL RULES:
1. If the context does not contain the answer to the question, you must reply exactly with: "I don't know." Do not invent, extrapolate, or use external knowledge.
2. For every fact or statement you make in your answer, you MUST cite the source file name using the marker format from the context (e.g., [handbook.md] or [policy.md]).

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

    return prompt
    

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

def answer_question(question):

    results = retrieve(question)

    prompt = build_prompt(
        question,
        results
    )

    answer = llm.invoke(prompt)

    return results, answer.content

In [15]:
questions = [
    "How long do I have to get a full refund?",
    "How do I reset my password?",
    "What is the company's stock price today?"    
]

for q in questions:

    results, answer = answer_question(q)

    print("="*50)
    print("QUESTION:", q)

    print("\nRETRIEVED SOURCES:")

    for meta in results["metadatas"][0]:
        print(meta["source"])

    print("\nANSWER:")
    print(answer)

QUESTION: How long do I have to get a full refund?

RETRIEVED SOURCES:
policy.md
policy.md
policy.md

ANSWER:
According to [policy.md], you have 30 days from the date of purchase to request a full refund, provided the item is unused and in its original packaging.
QUESTION: How do I reset my password?

RETRIEVED SOURCES:
it.md
handbook.md
policy.md

ANSWER:
According to [it.md], to reset your password, click 'Forgot password' on the login screen, and a reset link will be emailed to your registered address. The reset link expires after one hour for security.
QUESTION: What is the company's stock price today?

RETRIEVED SOURCES:
facilities.md
it.md
policy.md

ANSWER:
I don't know. [financials.md]
